# 00 - Get the Dataset and Run Quality Checks (MRL Eye Dataset)

**Phase 1, step 0.** Before cleaning or EDA we need (1) the data on disk and (2) a quick audit that tells us whether the data is usable.

**Part A: get the data** - pick one source in the settings cell:

| `SOURCE` | When to use |
|---|---|
| `'existing'` | The images are already in `DATA_DIR` (nothing is downloaded) |
| `'official'` | Download `mrlEyes_2018_01.zip` from the MRL website and unzip it |
| `'kaggle'` | Download a Kaggle copy (needs `kaggle.json`; set `KAGGLE_SLUG`) |
| `'upload'` | Google Colab: upload your own ZIP, or unzip a ZIP you already have (`LOCAL_ZIP`) |

**Part B: quality checks** - every check prints a result and is written to `data/processed/quality_report.json`:

1. Are there image files at all, and which extensions?
2. **Missing / malformed labels:** every filename must contain 8 fields; the *eye state* field must be 0 or 1.
3. Invalid values in the other coded fields (gender, glasses, reflections, lighting, sensor).
4. Unreadable or corrupt images.
5. Exact duplicate files (same bytes), and duplicates with *conflicting* labels.
6. Class balance and subject count (first look).

The detailed cleaning (removing bad rows) happens in `01_cleaning.ipynb`. This notebook only **reports** problems.

**MRL filename:** `s0001_00001_0_0_0_0_0_01.png` = `subject_imageID_gender_glasses_eyeState_reflections_lighting_sensor`; eye state **0 = closed, 1 = open**.

In [ ]:
import os, sys, json, hashlib, zipfile, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
IN_COLAB = 'google.colab' in sys.modules

# ---------------- settings ----------------
SOURCE      = 'existing'          # 'existing' | 'official' | 'kaggle' | 'upload'
DATA_DIR    = os.environ.get('MRL_DIR', '../data/raw/mrl')   # where the images live / will be stored
OFFICIAL_URL = 'http://mrl.cs.vsb.cz/data/eyedataset/mrlEyes_2018_01.zip'
KAGGLE_SLUG = ''                  # e.g. 'owner/dataset-name' (copy it from the Kaggle dataset page)
LOCAL_ZIP   = ''                  # path to a ZIP you already have (used by SOURCE='upload')
PROC_DIR, IMG_DIR = '../data/processed', '../images'
# ------------------------------------------
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(PROC_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
def savefig(name): plt.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches='tight')
IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp'}
def list_images(root):
    return [os.path.join(r, f) for r, _, fs in os.walk(root) for f in fs if os.path.splitext(f)[1].lower() in IMG_EXT]
print('Running on', 'Colab' if IN_COLAB else 'local Jupyter', '| DATA_DIR =', os.path.abspath(DATA_DIR))

## Part A. Get the data
Downloading is skipped automatically if `DATA_DIR` already contains images. ZIP files are extracted with a path check so that nothing can be written outside `DATA_DIR`.

In [ ]:
def safe_extract(zip_path, dest):
    dest = os.path.abspath(dest)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            target = os.path.abspath(os.path.join(dest, m.filename))
            if not target.startswith(dest + os.sep) and target != dest:
                raise RuntimeError(f'Unsafe path in ZIP: {m.filename}')
        z.extractall(dest)

def download(url, out):
    import requests
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(out, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc='downloading') as bar:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk); bar.update(len(chunk))

already = len(list_images(DATA_DIR))
if already:
    print(f'{already} images already in {DATA_DIR}: nothing to download.')
elif SOURCE == 'official':
    zip_path = os.path.join(os.path.dirname(os.path.abspath(DATA_DIR)), 'mrlEyes_2018_01.zip')
    download(OFFICIAL_URL, zip_path); safe_extract(zip_path, DATA_DIR)
elif SOURCE == 'kaggle':
    assert KAGGLE_SLUG, 'Set KAGGLE_SLUG first (the owner/name part of the Kaggle dataset URL).'
    if IN_COLAB and not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        from google.colab import files
        print('Upload kaggle.json'); files.upload()
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json')); os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', KAGGLE_SLUG, '-p', DATA_DIR, '--unzip'], check=True)
elif SOURCE == 'upload':
    zp = LOCAL_ZIP
    if not zp and IN_COLAB:
        from google.colab import files
        print('Choose the dataset .zip'); zp = list(files.upload())[0]
    assert zp and os.path.exists(zp), 'Set LOCAL_ZIP to an existing .zip file (or run on Colab to get an upload button).'
    safe_extract(zp, DATA_DIR)
else:
    print("SOURCE='existing' but no images found: put the images in DATA_DIR or choose another SOURCE.")

paths = list_images(DATA_DIR)
print(f'{len(paths):,} image files found under {DATA_DIR}')
assert paths, 'No images found. Check DATA_DIR / SOURCE (see the README for download links).'

## Part B. Quality checks
### B1. File inventory
Which extensions are present, and are there non-image files mixed in?

In [ ]:
report = {}
all_files = [os.path.join(r, f) for r, _, fs in os.walk(DATA_DIR) for f in fs]
ext = pd.Series([os.path.splitext(f)[1].lower() or '(none)' for f in all_files]).value_counts()
print(ext.to_string())
report['total_files'] = len(all_files); report['image_files'] = len(paths)
report['non_image_files'] = len(all_files) - len(paths)
report['extensions'] = ext.to_dict()
print(f"\n{report['image_files']:,} images, {report['non_image_files']} non-image files (ignored)")

### B2. Label and metadata check (missing / malformed / invalid)
The label is **inside the filename**, so a bad filename means a missing label. For each image we record whether the name has 8 fields, whether the eye state is `0`/`1`, and whether every other coded field is legal.

In [ ]:
COLS = ['subject', 'image_id', 'gender', 'glasses', 'eye_state', 'reflections', 'lighting', 'sensor']
ALLOWED = {'gender': {0, 1}, 'glasses': {0, 1}, 'eye_state': {0, 1}, 'reflections': {0, 1, 2}, 'lighting': {0, 1}, 'sensor': {1, 2, 3}}

rows = []
for p in paths:
    parts = os.path.splitext(os.path.basename(p))[0].strip().split('_')
    rec = {'path': p, 'n_fields': len(parts)}
    if len(parts) == 8: rec.update(dict(zip(COLS, parts)))
    rows.append(rec)
q = pd.DataFrame(rows)
for c in COLS:
    if c not in q: q[c] = np.nan
for c in COLS[1:]: q[c] = pd.to_numeric(q[c], errors='coerce')

q['bad_format']   = q.n_fields != 8
q['missing_label'] = q.eye_state.isna()
q['bad_label']     = q.eye_state.notna() & ~q.eye_state.isin([0, 1])
q['bad_subject']   = ~q.subject.astype(str).str.lower().str.match(r'^s\d+$')
q['bad_values']    = pd.concat([(q[c].notna() & ~q[c].isin(ok)) for c, ok in ALLOWED.items() if c != 'eye_state'], axis=1).any(axis=1)

checks = q[['bad_format', 'missing_label', 'bad_label', 'bad_subject', 'bad_values']].sum().astype(int)
print('Problem counts (rows):'); print(checks.to_string())
print('\nMissing values per field:'); print(q[COLS].isna().sum().to_string())
report['label_checks'] = checks.to_dict()
report['missing_per_field'] = q[COLS].isna().sum().astype(int).to_dict()

bad = q[q[['bad_format', 'missing_label', 'bad_label', 'bad_subject', 'bad_values']].any(axis=1)]
print(f'\n{len(bad)} images with a label/metadata problem.')
if len(bad): print(bad.path.map(os.path.basename).head(10).to_string(index=False))

### B3. Unreadable / corrupt images, sizes and colour modes
Each image is opened and fully decoded. We also collect the file hash (for duplicates) and the image size / mode (to see format inconsistencies).

In [ ]:
def inspect(p):
    rec = {'readable': True}
    try:
        with open(p, 'rb') as f: raw = f.read()
        rec['md5'] = hashlib.md5(raw).hexdigest(); rec['kb'] = len(raw) / 1024
        with Image.open(p) as im:
            rec['mode'] = im.mode; rec['w'], rec['h'] = im.size
            im.load()                           # forces full decode -> catches truncated files
    except Exception as e:
        rec['readable'] = False; rec['error'] = str(e)[:80]
    return rec

q = pd.concat([q, pd.DataFrame([inspect(p) for p in tqdm(q.path)])], axis=1)
print('Unreadable / corrupt images:', int((~q.readable).sum()))
if (~q.readable).any(): print(q.loc[~q.readable, ['path', 'error']].head().to_string(index=False))
ok = q[q.readable]
print('\nColour modes:', ok['mode'].value_counts().to_dict())
print('Distinct image sizes:', ok.groupby(['w', 'h']).ngroups, '| width', ok.w.min(), '-', ok.w.max(), '| height', ok.h.min(), '-', ok.h.max())
report['unreadable'] = int((~q.readable).sum()); report['color_modes'] = ok['mode'].value_counts().to_dict()
report['distinct_sizes'] = int(ok.groupby(['w', 'h']).ngroups)

### B4. Duplicates
Identical files are risky: if a copy lands in the test set while the original is in training, the test score is inflated. If the copies carry **different labels**, at least one label is wrong.

In [ ]:
dup_rows = int(ok.duplicated('md5').sum())
conf = ok.groupby('md5').eye_state.nunique(); conflicts = int((conf > 1).sum())
print('Exact duplicate files (extra copies):', dup_rows)
print('Duplicate groups with conflicting labels:', conflicts)
report['duplicate_files'] = dup_rows; report['duplicate_groups_conflicting_labels'] = conflicts

### B5. First look at class balance and subjects

In [ ]:
good = q[~q[['bad_format', 'missing_label', 'bad_label', 'bad_subject', 'bad_values']].any(axis=1) & q.readable].copy()
good['label'] = good.eye_state.map({0: 'closed', 1: 'open'})
cnt = good.label.value_counts(); ratio = cnt.max() / cnt.min()
print(cnt.to_string()); print(f'imbalance ratio (majority/minority): {ratio:.2f}')
print('Subjects:', good.subject.nunique(), '| images per subject: min', good.groupby('subject').size().min(), 'max', good.groupby('subject').size().max())
report['usable_images'] = int(len(good)); report['class_counts'] = cnt.to_dict(); report['subjects'] = int(good.subject.nunique())
report['imbalance_ratio'] = float(ratio)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=good, x='label', order=['open', 'closed'], palette=['#4c9f70', '#d9534f'], ax=ax[0]); ax[0].set_title('Class balance')
for c in ax[0].containers: ax[0].bar_label(c)
good.groupby('subject').size().plot.hist(bins=25, ax=ax[1], color='slateblue'); ax[1].set_title('Images per subject')
q.loc[q.readable, 'w'].plot.hist(bins=30, ax=ax[2], color='steelblue'); ax[2].set_title('Image width (px)')
plt.tight_layout(); savefig('quality_01_overview.png'); plt.show()

fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for r, lab in enumerate(['open', 'closed']):
    for c, p in enumerate(good[good.label == lab].sample(8, random_state=1).path):
        ax[r, c].imshow(Image.open(p).convert('L'), cmap='gray'); ax[r, c].axis('off')
    ax[r, 0].set_title(lab, loc='left')
plt.suptitle('Random images per class (sanity check that labels look right)'); plt.tight_layout(); savefig('quality_02_samples.png'); plt.show()

### B6. Verdict
The report is saved so it can be quoted in the README. Hard failures (no usable images, only one class) stop the notebook; everything else is handled in `01_cleaning.ipynb`.

In [ ]:
json.dump(report, open(f'{PROC_DIR}/quality_report.json', 'w'), indent=2, default=int)
print(json.dumps(report, indent=2, default=int))

assert report['usable_images'] > 0, 'No usable images: every file has a label/metadata problem or is unreadable.'
assert len(cnt) == 2, 'Only one class present: both open and closed eyes are required.'
problems = report['unreadable'] + report['duplicate_files'] + sum(report['label_checks'].values())
print('\nVERDICT:', 'data is clean enough to continue' if problems == 0 else f'{problems} issues found -> they are removed/handled in 01_cleaning.ipynb')

**What this tells us:** the counts above show how many labels are missing or malformed, how many files are corrupt or duplicated, and whether the two classes are balanced. This is the evidence for the cleaning decisions in the next notebook.

**Next:** `01_cleaning.ipynb`